# 🚆 สถิติผู้โดยสารระบบขนส่งสาธารณะในประเทศไทย
**วิเคราะห์ปริมาณการเดินทางของประชาชนด้วยระบบขนส่งสาธารณะ**

**ชุดข้อมูล:** สถิติผู้โดยสารรายวันของระบบขนส่งสาธารณะทั่วประเทศไทย (ปี 2568–2569)  
**ช่วงเวลา:** ประมาณ 14 เดือน  
**แหล่งข้อมูล:** กระทรวงคมนาคม

---

## วัตถุประสงค์การวิเคราะห์
1. **สัดส่วนการใช้ระบบขนส่ง (Modal Share)** — ระบบใดมีผู้โดยสารมากที่สุด?
2. **การเปรียบเทียบรถไฟฟ้าในเมือง (Urban Rail Comparison)** — พฤติกรรมผู้โดยสารแต่ละสายแตกต่างกันอย่างไร?
3. **การตรวจจับเหตุการณ์พิเศษ (Event Detection)** — วันหยุดและเทศกาลปรากฏในข้อมูลหรือไม่?
4. **การพยากรณ์ (Forecasting)** — คาดการณ์ความต้องการผู้โดยสาร 30 วันข้างหน้าด้วย Facebook Prophet

In [3]:
# ติดตั้งแพ็กเกจที่จำเป็น (รันครั้งเดียวใน Colab)
!pip install prophet plotly scikit-learn -q

zsh:1: command not found: pip


In [4]:
# นำเข้าไลบรารีทั้งหมดที่ใช้ในการวิเคราะห์
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from scipy import stats
from sklearn.metrics import mean_absolute_error, mean_squared_error
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
import warnings
warnings.filterwarnings('ignore')

print('✅ นำเข้าไลบรารีสำเร็จ')

/Users/sarat/Code/superai-se6/superai-se6-mini-hackathon-1/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ นำเข้าไลบรารีสำเร็จ


---
## เฟส 1 — โหลดข้อมูล (Data Loading)

### เป้าหมาย
โหลดชุดข้อมูลและตรวจสอบโครงสร้างเบื้องต้น

**ชุดข้อมูลที่ใช้:**
- `passengers68.csv` — ข้อมูลผู้โดยสารปี 2568
- `passengers69.csv` — ข้อมูลผู้โดยสารปี 2569

In [12]:
# โหลดชุดข้อมูลทั้งสองไฟล์
df68 = pd.read_csv('https://raw.githubusercontent.com/lumduan/superai-se6-mini-hackathon-1/main/dataset-5/data/passengers68.csv')
df69 = pd.read_csv('https://raw.githubusercontent.com/lumduan/superai-se6-mini-hackathon-1/main/dataset-5/data/passengers69.csv')

# รวมข้อมูลเป็น DataFrame เดียว
df = pd.concat([df68, df69], ignore_index=True)

print(f'df68 จำนวนแถว: {df68.shape[0]:,}  คอลัมน์: {df68.shape[1]}')
print(f'df69 จำนวนแถว: {df69.shape[0]:,}  คอลัมน์: {df69.shape[1]}')
print(f'รวมทั้งหมด:   {df.shape[0]:,}  คอลัมน์: {df.shape[1]}')
df.head()

df68 จำนวนแถว: 69,440  คอลัมน์: 8
df69 จำนวนแถว: 3,010  คอลัมน์: 8
รวมทั้งหมด:   72,450  คอลัมน์: 8


,รูปแบบการเดินทาง,วัตถุประสงค์,สาธารณะ/ส่วนบุคคล,หน่วยงาน,ยานพาหนะ/ท่า,วันที่,หน่วย,ปริมาณ
0,ทางถนน,การเดินทางระหว่างจังหวัด,สาธารณะ,บขส.,รถ บขส. และ รถร่วม,01/01/2025,คน,"127,551"
1,ทางถนน,การเดินทางระหว่างจังหวัด,สาธารณะ,ขบ.,รถหมวด 3,01/01/2025,คน,"8,218"
2,ทางถนน,การเดินทางระหว่างจังหวัด,ส่วนบุคคล,ทล.,รถยนต์เฉพาะ 4 ล้อ (10 จุดสำรวจ),01/01/2025,คัน,"877,943"
3,ทางถนน,การเดินทางระหว่างจังหวัด,ส่วนบุคคล,ทล.,รถยนต์ทุกประเภท (10 จุดสำรวจ),01/01/2025,คัน,"932,642"
4,ทางถนน,การเดินทางระหว่างจังหวัด,ส่วนบุคคล,กทพ.,รถยนต์เฉพาะ 4 ล้อ (ทางด่วน),01/01/2025,คัน,"1,364,992"


In [14]:
from IPython.display import display
import pandas as pd

print("=== ข้อมูลโครงสร้าง DataFrame ===")

display(pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.values
}))

=== ข้อมูลโครงสร้าง DataFrame ===


,column,dtype
0,รูปแบบการเดินทาง,str
1,วัตถุประสงค์,str
2,สาธารณะ/ส่วนบุคคล,str
3,หน่วยงาน,str
4,ยานพาหนะ/ท่า,str
5,วันที่,str
6,หน่วย,str
7,ปริมาณ,str


---
## เฟส 2 — ทำความสะอาดและตรวจสอบข้อมูล (Data Cleaning & Validation)

### ขั้นตอน
1. แปลงคอลัมน์วันที่เป็น datetime
2. จัดเรียงข้อมูลตามวันที่
3. ตรวจสอบคุณภาพข้อมูล (ค่าว่าง, ซ้ำซ้อน, ค่าติดลบ, สถิติเบื้องต้น)

In [15]:
# แปลงคอลัมน์วันที่และจัดเรียงข้อมูล
df['date'] = pd.to_datetime(df['วันที่'], dayfirst=True)
df = df.sort_values('date').reset_index(drop=True)

print(f'ช่วงวันที่: {df["date"].min().date()} → {df["date"].max().date()}')
print(f'จำนวนวันทั้งหมด: {(df["date"].max() - df["date"].min()).days + 1} วัน')

ช่วงวันที่: 2025-01-01 → 2026-03-11
จำนวนวันทั้งหมด: 435 วัน


In [ ]:
print("📊 รายงานคุณภาพข้อมูล (Data Quality Report)")

# =====================================================
# 1️⃣ Missing Values
# =====================================================
print("1️⃣ Missing Values ต่อคอลัมน์")

missing_df = (
    df.isna()
    .sum()
    .reset_index()
)

missing_df.columns = ["column", "missing_values"]

display(missing_df)


# =====================================================
# 2️⃣ Duplicate Rows
# =====================================================
print("2️⃣ Duplicate Rows")

duplicate_count = df.duplicated().sum()

duplicate_df = pd.DataFrame({
    "metric": ["duplicate_rows"],
    "value": [duplicate_count]
})

duplicate_df["value"] = duplicate_df["value"].apply(lambda x: f"{x:,}")

display(duplicate_df)

# ลบ duplicate
df = df.drop_duplicates()


# =====================================================
# 3️⃣ Convert Passenger Column to Numeric
# =====================================================
# บาง dataset มี comma เช่น "1,234,567"
# ต้องลบ comma ก่อนแปลงเป็นตัวเลข

df["ปริมาณ"] = pd.to_numeric(
    df["ปริมาณ"].astype(str).str.replace(",", ""),
    errors="coerce"
)


# =====================================================
# 4️⃣ Negative Value Check
# =====================================================
print("3️⃣ Negative Values in 'ปริมาณ'")

neg_mask = df["ปริมาณ"] < 0
negative_count = neg_mask.sum()

negative_df = pd.DataFrame({
    "metric": ["negative_values"],
    "value": [negative_count]
})

negative_df["value"] = negative_df["value"].apply(lambda x: f"{x:,}")

display(negative_df)

# ลบค่าติดลบ (ไม่ควรมีใน passenger count)
df = df[~neg_mask]


# =====================================================
# 5️⃣ Descriptive Statistics
# =====================================================
print("4️⃣ Passenger Statistics (คอลัมน์: ปริมาณ)")

stats = df["ปริมาณ"].describe()

stats_df = pd.DataFrame({
    "metric": [
        "จำนวนข้อมูล (count)",
        "ค่าเฉลี่ยผู้โดยสารต่อวัน (mean)",
        "ส่วนเบี่ยงเบนมาตรฐาน (std)",
        "ค่าน้อยที่สุด (min)",
        "เปอร์เซ็นไทล์ 25%",
        "ค่ากลาง / มัธยฐาน (median)",
        "เปอร์เซ็นไทล์ 75%",
        "ค่ามากที่สุด (max)"
    ],
    "value": [
        stats["count"],
        stats["mean"],
        stats["std"],
        stats["min"],
        stats["25%"],
        stats["50%"],
        stats["75%"],
        stats["max"]
    ]
})

# format ตัวเลขให้อ่านง่าย
stats_df["value"] = stats_df["value"].apply(lambda x: f"{x:,.0f}")

display(stats_df)


print("""
📌 Interpretation:

- ค่าเฉลี่ยผู้โดยสารต่อวัน ~ 166k คน
- มัธยฐาน ~ 21k คน → distribution skewed มาก
- ค่าสูงสุด ~ หลายล้าน → น่าจะเป็นระบบถนนหรือทางด่วน
- ค่า std สูงกว่า mean มาก → dataset มีความกระจายสูง
""")

📊 รายงานคุณภาพข้อมูล (Data Quality Report)
1️⃣ Missing Values ต่อคอลัมน์


,column,missing_values
0,รูปแบบการเดินทาง,1
1,วัตถุประสงค์,1
2,สาธารณะ/ส่วนบุคคล,1
3,หน่วยงาน,1
4,ยานพาหนะ/ท่า,1
5,วันที่,1
6,หน่วย,1
7,ปริมาณ,445
8,date,1


2️⃣ Duplicate Rows


,metric,value
0,duplicate_rows,0


3️⃣ Negative Values in 'ปริมาณ'


,metric,value
0,negative_values,0


4️⃣ Passenger Statistics (คอลัมน์: ปริมาณ)


,metric,value
0,จำนวนข้อมูล (count),"18,261"
1,ค่าเฉลี่ยผู้โดยสารต่อวัน (mean),"166,410"
2,ส่วนเบี่ยงเบนมาตรฐาน (std),"367,846"
3,ค่าน้อยที่สุด (min),0
4,เปอร์เซ็นไทล์ 25%,"2,206"
5,ค่ากลาง / มัธยฐาน (median),"21,863"
6,เปอร์เซ็นไทล์ 75%,"76,848"
7,ค่ามากที่สุด (max),"3,509,191"



📌 Interpretation:

- ค่าเฉลี่ยผู้โดยสารต่อวัน ~ 166k คน
- มัธยฐาน ~ 21k คน → distribution skewed มาก
- ค่าสูงสุด ~ หลายล้าน → น่าจะเป็นระบบถนนหรือทางด่วน
- ค่า std สูงกว่า mean มาก → dataset มีความกระจายสูง

